In [6]:
import os
# hugging face镜像设置，如果国内环境无法使用启用该设置
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
load_dotenv()

markdown_path = "../../data/C1/markdown/easy-rl-chapter1.md"

# 加载本地markdown文件
loader = UnstructuredMarkdownLoader(markdown_path)
docs = loader.load()

# 文本分块
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

# 中文嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
  
# 构建向量存储
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents(chunks)

# 提示词模板
prompt = ChatPromptTemplate.from_template("""请根据下面提供的上下文信息来回答问题。
请确保你的回答完全基于这些上下文。
如果上下文中没有足够的信息来回答问题，请直接告知：“抱歉，我无法根据提供的上下文找到相关信息来回答此问题。”

上下文:
{context}

问题: {question}

回答:"""
                                          )

# 配置大语言模型

# 使用 AIHubmix
# llm = ChatOpenAI(
#     model="glm-4.7-flash-free",
#     temperature=0.7,
#     max_tokens=4096,
#     api_key=os.getenv("DEEPSEEK_API_KEY"),
#     base_url="https://aihubmix.com/v1"
# )

llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0.7,
    max_tokens=2048,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# 用户查询
question = "文中举了哪些例子？"

# 在向量存储中查询相关文档
retrieved_docs = vectorstore.similarity_search(question, k=3)
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

answer = llm.invoke(prompt.format(question=question, context=docs_content))
print(answer)


content='根据上下文，文中举的例子包括：\n\n（1）DeepMind 研发的走路的智能体，学习在曲折道路上保持平衡前进。  \n（2）探索和利用的例子：选择餐馆（利用已知喜欢的餐馆 vs 探索新餐馆）、做广告（利用最优策略 vs 探索新策略）、挖油（在已知地点挖油 vs 在新地点探索）、玩游戏（如《街头霸王》中固定出脚策略 vs 尝试新策略）。  \n（3）强化学习的例子：AlphaGo 击败人类棋手，以及自然界中羚羊通过试错学习站立和奔跑。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 565, 'total_tokens': 689, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 64}, 'prompt_cache_hit_tokens': 64, 'prompt_cache_miss_tokens': 501}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '73e8de9d-4f5f-4126-9c89-db5509e5c922', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--8a31c0db-e7b2-4bca-b6e6-dce8ce43c6a8-0' usage_metadata={'input_tokens': 565, 'output_tokens': 124, 'total_tokens': 689, 'input_token_details': {'cache_read': 64}, 'output_token_details': {}}


In [7]:
from IPython.display import Markdown
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
result = parser.invoke(answer)
print(result)

根据上下文，文中举的例子包括：

（1）DeepMind 研发的走路的智能体，学习在曲折道路上保持平衡前进。  
（2）探索和利用的例子：选择餐馆（利用已知喜欢的餐馆 vs 探索新餐馆）、做广告（利用最优策略 vs 探索新策略）、挖油（在已知地点挖油 vs 在新地点探索）、玩游戏（如《街头霸王》中固定出脚策略 vs 尝试新策略）。  
（3）强化学习的例子：AlphaGo 击败人类棋手，以及自然界中羚羊通过试错学习站立和奔跑。
